# Segmentación de clientes

In [ ]:
# importando librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore") 

In [ ]:
# cargando dataset
df = pd.read_csv("../data/Customer Data.csv")
df.head()

In [ ]:
# visualizando informacion del dataframe
df.info()

In [ ]:
# estadística descriptiva
df.describe().T

In [ ]:
# visualizando valores nulos
df.isnull().sum()

In [ ]:
# rellenando los valores nulos con el promedio de la columna
df["MINIMUM_PAYMENTS"] = df["MINIMUM_PAYMENTS"].fillna(df["MINIMUM_PAYMENTS"].mean())
df["CREDIT_LIMIT"] = df["CREDIT_LIMIT"].fillna(df["CREDIT_LIMIT"].mean())

In [ ]:
# revisando si existen filas duplicadas
df.duplicated().sum()

In [ ]:
# eliminando la columna CUST_ID
df.drop(columns=["CUST_ID"],axis=1,inplace=True)

In [ ]:
# visualizando la distribuci?n de los datos
plt.figure(figsize=(10,60))
for i in range(df.shape[1]):
    plt.subplot(df.shape[1], 1, i + 1)
    sns.histplot(df.iloc[:, i], kde=True, color='g', alpha=0.6)
    plt.title(df.columns[i])
plt.tight_layout()


In [ ]:
# mapa de calor
plt.figure(figsize=(20, 12)) # creando lienzo
matrix = np.triu(df.corr()) # creando matriz
sns.heatmap(df.corr(), annot=True,linewidth=.8, mask=matrix, cmap="coolwarm",cbar=False) # creando visual

In [ ]:
# escalando el dataframe
scalar=StandardScaler()
scaled_df = scalar.fit_transform(df)

In [ ]:
# reduccion de dimensionalidad (PCA), se realiza para llevar a dos dimensiones las caracteristicas
pca = PCA(n_components=2)
principal_components = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(data=principal_components ,columns=["PCA1","PCA2"])
pca_df.head()

In [ ]:
# buscando el valor de k para el modelo de k means
inertia = []
range_val = range(1,15)
for i in range_val:
    kmean = KMeans(n_clusters=i, random_state=42)
    kmean.fit_predict(pd.DataFrame(scaled_df))
    inertia.append(kmean.inertia_)
plt.plot(range_val,inertia,'b-o')
plt.xlabel('Valores de K') 
plt.ylabel('Inercia') 
plt.title('Método Elbow usando Inercia') 
plt.show()

In [ ]:
# contruyendo el modelo de k means
kmeans_model=KMeans(4, random_state=42)
kmeans_model.fit_predict(scaled_df)
pca_df_kmeans= pd.concat([pca_df,pd.DataFrame({'cluster':kmeans_model.labels_})],axis=1)

In [ ]:
# visualizando los cluster
plt.figure(figsize=(8,8))
ax=sns.scatterplot(x="PCA1",y="PCA2",hue="cluster",data=pca_df_kmeans,palette=['red','green','blue','black'])
plt.title("Clustering usando K-Means")
plt.show()

In [ ]:
# encontrando los centroides
cluster_centers = pd.DataFrame(data=kmeans_model.cluster_centers_,columns=df.columns)
# revirtiendo la escalarizacion
cluster_centers = scalar.inverse_transform(cluster_centers)
cluster_centers = pd.DataFrame(data=cluster_centers,columns=df.columns)
cluster_centers.head()

In [ ]:
# pegando la variable las predicciones al cluster original
cluster_df = pd.concat([df,pd.DataFrame({'Cluster':kmeans_model.labels_})],axis=1)
cluster_df.head()

In [ ]:
# visualizando cluster 0
cluster_1_df = cluster_df[cluster_df["Cluster"]==0]
cluster_1_df.head()

In [ ]:
# visualizando la segmentacion
sns.countplot(x='Cluster', data=cluster_df)
plt.show()

In [ ]:
# visualizando la distribucion de las variables por cluster
for c in cluster_df.drop(['Cluster'],axis=1):
    grid= sns.FacetGrid(cluster_df, col='Cluster')
    grid= grid.map(plt.hist, c)
plt.show()

In [ ]:
# visualizando el promedio de PAYMENTS por cluster
df1 = cluster_df.groupby(['Cluster'])['PAYMENTS'].mean().reset_index().sort_values(by='PAYMENTS', ascending=False)
df1.head()